In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <호출(invoke) 출력>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 오픈AI의 대규모 언어 모델 설정
model = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 주어진 주제에 대한 설명 요청
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧게 설명을 해주세요")
# 출력 파서 정의: AI 메시지의 출력 내용을 추출
parser = StrOutputParser()
# 프롬프트, 모델, 출력 파서를 체인으로 연결
chain = prompt | model | parser

# 응답을 호출
chain.invoke({"topic":"더블딥"})


'더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 두 번 발생하는 상황을 지칭합니다. 첫 번째 경기가 회복되기 전에 다시 또 다른 경기 침체가 오는 경우를 말하며, 이는 경제가 회복세에 접어들었다가 다시 불황으로 빠지는 복잡한 회복 과정을 나타냅니다. 더블딥은 통상적인 경기 사이클을 넘어, 일시적인 회복 신호가 나타났다가 다시 경제적 압력이 가중되는 경우에 해당합니다. 이러한 상황은 정책 결정자나 경제 분석가에게 큰 도전 과제가 될 수 있습니다.'

In [7]:
# <배치(Batch) 출력>

# 주어진 주제 리스트를 배치로 출력
chain.batch([{"topic": "더블딥"}, {"topic": "인플레이션"}])

['더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 한번 발생한 후 일시적으로 회복되다가 다시 침체에 빠지는 상황을 의미합니다. 즉, 경제가 V자 형태로 회복하지 않고 W자 형태를 취하는 경우를 나타냅니다. 이러한 현상은 소비자 신뢰 부족, 기업 투자 감소, 정책의 효과 미비 등 다양한 요인에 의해 발생할 수 있습니다. 더블딥은 경제와 금융 시장에 큰 영향을 미치며, 정책 결정자들에게도 중요한 분석 대상이 됩니다.',
 '인플레이션은 일반적인 물가 상승을 의미하며, 이는 경제 내에서 상품과 서비스의 가격이 지속적으로 오르는 현상입니다. 인플레이션이 발생하면, 화폐의 구매력이 감소하게 되어 동일한 금액으로 구매할 수 있는 상품과 서비스의 수가 줄어듭니다. 주로 수요 증가, 생산비 상승, 통화 공급 증가 등이 인플레이션의 원인으로 작용할 수 있습니다. 적정 수준의 인플레이션은 경제 성장의 신호로 여겨지기도 하지만, 지나치면 경제에 부정적인 영향을 미칠 수 있습니다.']

In [8]:
# <스트림(Stream) 출력>

# 응답을 토큰 단위로 스트리밍하여 출력
for token in chain.stream({"topic":"더블딥"}):
    # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며, 버퍼를 즉시 플러시하여 실시간으로 보여줌
    print(token, end="", flush=True)


더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 발생한 후 일시적으로 회복되는 듯하다가 다시 경기 침체에 접어드는 현상을 의미합니다. 이 현상은 경제가 두 번의 침체를 겪는 것으로, 첫 번째 침체 후 개선 징후가 나타나지만, 여러 가지 요인으로 인해 다시 하락세로 돌아가는 상황을 설명합니다. 주로 금융 위기나 글로벌 경제 불안정성 등으로 인해 발생할 수 있으며, 경제 정책의 효과나 소비자 신뢰도에도 영향을 받습니다.

In [9]:
# <구성된 체인을 다른 러너블과 결합하기>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# "이 대답을 영어로 번역해 주세요"라는 질문을 생성하는 프롬프트 템플릿을 정의
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 이전에 정의된 체인(chain)을 사용하여 대답을 생성하고, 그 대답을 영어로 번역하도록 프롬프트에 전달한 후, 모델을 통해 결과를 생성하여 최종적으로 문자열로 파싱하는 체인을 구성
composed_chain = {"answer": chain} | analysis_prompt | model | StrOutputParser()
# "더블딥"이라는 주제로 대답을 생성하고, 체인 실행
composed_chain.invoke({"topic": "더블딥"})

'"Double Dip" is a term primarily used in economics that refers to a phenomenon where an economy temporarily recovers after a recession, only to fall back into a downturn. In this case, the economy shows a rapid recovery resembling a "V" shape, followed by experiencing a second trough that resembles a "W" shape. Double dips raise significant concerns for investors and policymakers, indicating uncertainty in the economic recovery. This phenomenon can mainly occur due to fundamental issues within the economy or external shocks.'

In [10]:
# <람다 함수를 사용한 체인을 통해 구성하기>

# 이전에 정의된 값들
model = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
chain = prompt | model | StrOutputParser()
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해 주세요: {answer}")

# 람다 함수를 사용한 체인 구성
composed_chain_with_lambda = (
    # 이전에 정의된 체인(chain)을 사용하여 입력된 데이터를 받아옵니다.
    chain
    # 입력된 데이터를 "answer" 키로 변환하는 람다 함수를 적용합니다.
    | (lambda input: {"answer": input})
    # "answer " 키를 가진 데이터를 영어로 번역하도록 프롬프트에 전달합니다.
    | analysis_prompt
    # 프롬프트에서 생성된 요청을 모델에 전달하여 결과를 생성합니다.
    | model
    # 모델에서 반환된 결과를 문자열로 파싱합니다.
    | StrOutputParser()
)
# "더블딥"라는 주제로 대답을 생성하고, 그 대답을 영어로 번역합니다.
composed_chain_with_lambda.invoke({"topic": "더블딥"})


'"Double Dip" is a term used in economics to describe a situation where a recession occurs, and after failing to recover, the economy falls into another recession. In other words, it refers to a scenario where the economy hits a low point and rebounds, but that rebound does not last, leading to another low point. This phenomenon can occur due to factors such as declining consumer confidence, reduced investment, and decreased employment, and it represents the uncertainty of economic recovery.'

In [11]:
# <`.pipe()`를 통해 체인 구성하기>

# (방법1) 여러 작업을 순차적으로 .pipe를 통해 연결하여 체인 구성하기
composed_chain_with_pipe = (
  # 이전에 정의된 체인(chain)으로 입력된 데이터를 받아옴
  chain
  # 입력된 데이터를 "answer" 키로 변환하는 람다 함수 적용
  .pipe(lambda input: {"answer": input})
  # analysis_prompt를 체인에 연결하여 설명을 영어로 번역하는 작업 추가
  .pipe(analysis_prompt)
  # 모델을 사용해 응답 생성
  .pipe(model)
  # 생성된 응답을 문자열로 파싱
  .pipe(StrOutputParser())
)
# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})


'"Double Dip" is a term primarily used in economics and finance that refers to the phenomenon where the economy falls back into recession after experiencing a recovery. Generally, this occurs after a certain period of recovery following the initial decline, but that recovery is incomplete or fragile, leading to another economic downturn. This concept is often analyzed through changes in indicators such as economic growth rates, employment figures, and consumer confidence.'

In [12]:
# (방법2) 좀 더 간단하게 연결하기
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())

# "더블딥"이라는 주제로 체인을 실행하여 답변 생성
composed_chain_with_pipe.invoke({"topic": "더블딥"})

'"Double Dip" is a term used in the stock market or economy to describe the phenomenon where the price of a particular asset falls sharply, rebounds, and then declines again. It is often interpreted as a signal of recession or market instability and can cause confusion among investors. This pattern can be triggered by psychological factors or external economic influences.'

In [13]:
# <`RunnableParallel`을 이용한 체인 구성>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧은 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥(Double deep)은 딥러닝의 한 종류로, 일반적인 딥러닝에서보다 더 심층적인 구조를 가진 신경망을 의미합니다. 이는 더 많은 층(layer)을 가지고 있어 더 복잡한 문제에 대한 해결이 가능하며, 높은 수준의 성능을 보장할 수 있습니다. 하지만 레이어가 많은 만큼 학습이 어렵고 복잡해지는 단점이 있습니다. 최근에는 더블딥을 이용한 다양한 연구와 응용이 활발히 진행되고 있습니다.
영어 설명: The double deep method is a type of storage configuration in a warehouse where pallets are stored two deep on either side of a double-sided aisle. This method maximizes storage capacity while still allowing for easy access to inventory.


In [14]:
# <`RunnableParallel`를 이용한 체인 구성하기>
from langchain_core.runnables import RunnableParallel
# OpenAI 모델 초기화
model = ChatOpenAI()
# 한국어 설명 생성 프롬프트 체인
kor_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 영어 설명 생성 프롬프트 체인
eng_chain = (
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명을 해주세요.")
    | model
    | StrOutputParser()
)
# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
# 주제에 대한 한국어와 영어 설명 생성
result = parallel_chain.invoke({"topic": "더블딥"})
# 결과 출력
print("한글 설명:", result['kor'])
print("영어 설명:", result['eng'])


한글 설명: 더블딥(Double deep)은 딥러닝 모델의 변종 중 하나로, 두 개의 딥러닝 모델을 같이 학습시켜 성능을 향상시키는 방법을 말합니다. 이 과정에서 한 모델은 다른 모델의 예측 오류를 보정하고, 그 반대의 역할을 수행하여 모델의 성능을 최적화합니다.더블딥은 이미지 분류, 자연어 처리 등 다양한 분야에서 사용되고 있습니다.
영어 설명: Double-dipping is the act of dipping a chip or food item into a communal dip after taking a bite, which is considered rude or unhygienic in some social settings.


In [15]:
# 참고
class CustomLCEL:
    def __init__(self, value):
        self.value = value  # 객체 생성 시 값을 초기화합니다.
    def __or__(self, other):
        if callable(other):
            return CustomLCEL(other(self.value))  # other가 함수일 경우, 함수를 호출하고 그 결과를 새로운 객체로 반환합니다.
        else:
            raise ValueError("Right operand must be callable")  # other가 함수가 아니면 에러를 발생시킵니다.
    def result(self):
        return self.value  # 현재 값을 반환합니다.
# 문자열 끝에 느낌표를 추가하는 함수
def add_exclamation(s):
    return s + "!"
# 문자열을 뒤집는 함수
def reverse_string(s):
    return s[::-1]
# 파이프라인을 생성하여 순차적으로 문자열 변환 작업을 수행합니다.
custom_chain = (
    CustomLCEL("랭체인 공부하기")  # "랭체인 공부하기"로 초기화된 객체 생성
    | add_exclamation  # 느낌표 추가
    | reverse_string  # 문자열 뒤집기
)
# 최종 결과를 출력합니다.
result = custom_chain.result()
print(result)
# 출력: !기하부공 인체랭


!기하부공 인체랭
